# A.4 Expressions

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Various items can be combined in AMPL's arithmetic and logical expressions. An expression
that may not contain variables is denoted cexpr and is sometimes called a "constant expression",

```ampl
Precedence     Name                 Type   Remarks
  1         if-then-else            A, S   A: if no else, then "else 0"
                                                 assumed e sexpr" required
  2         or ||                   L
  3         exists forall           L      logical reduction operators
  4         and &&                  L
  5         < <= = == <> != >= >    L
  6         in not in               L      membership in set
  6         within not within       L      S within T means set S ⊆ set T
  7         not !                   L      logical negation
  8         union diff symdiff      S      symdiff ≡ symmetric difference
  9         inter                   S      set intersection
 10         cross                   S      cross or Cartesian product
 11         setof .. by             S      set constructors
 12         + - less                A      a less b ≡ max (a − b, 0 )
 13         sum prod min max        A      arithmetic reduction operators
 14         * / div mod             A      div ≡ truncated quotient of integers
 15         + -                     A      unary plus, unary minus
 16         ˆ **                    A      exponentiation

  Operators are listed in increasing precedence order.
  Exponentiation and if-thenelse are right-associative;
       the other operators are left-associative.
  The 'Type' column indicates result types:
       A for arithmetic, L for logical, S for set.
```

Table A-1: Arithmetic, logical and set operators.

even though it may involve dummy indices. A logical expression, denoted lexpr, may not contain
variables when it is part of a cexpr. Set expressions are denoted sexpr.

Table A-1 summarizes the arithmetic, logical and set operators; the type column indicates
whether the operator produces an arithmetic value (A), a logical value (L), or a set value (S).

Arithmetic expressions are formed from the usual arithmetic operators, built-in functions, and
arithmetic reduction operators like sum:

```ampl
expr:
	number
	variable
	expr arith - op expr             arith - op is + - less * / mod div ˆ \*\*
	unary - op expr                  unary - op is + -
	built - in( exprlist )
	if lexpr then expr [ else expr ]
	reduction - op indexing expr     reduction - op is sum prod max min
	( expr )
```

Built-in functions are listed in Table A-2.

The arithmetic reduction operators are used in expressions like

```ampl
sum {i in Prod} cost[i] * Make[i]
```

The scope of the indexing expression extends to the end of the expr. If the operation is over an
empty set, the result is the identity value for the operation: 0 for sum, 1 for prod, Infinity for
min, and -Infinity for max.
Logical expressions appear where a value of "true" or "false" is required: in `check` statements,
the "such that" parts of indexing expressions (following the colon), and in if lexpr
then ... else ... expressions. Numeric values that appear in any of these contexts are implicitly coerced
to logical values: 0 is interpreted as false, and all other numeric values as true.

```ampl
lexpr:
      expr
      expr compare - op expr    compare - op is < <= = == != <> > >=
      lexpr logic - op lexpr    logic - op is or || and &&
      not lexpr
      member in sexpr
      member not in sexpr
      sexpr within sexpr
      sexpr not within sexpr
      opname indexing lexpr     opname is exists or forall
      ( lexpr )
```

The in operator tests set membership. Its left operand is a potential set member, i.e., an
expression or comma-separated list of expressions enclosed in parentheses, with the number of
expressions equal to the dimension of the right operand, which must be a set expression. The
within operator tests whether one set is contained in another. The two set operands must have
the same dimension.

The logical reduction operators exists and forall are the iterated counterparts of or and
and. When applied over an empty set, exists returns false and forall returns true.

Set expressions yield sets.

```ampl
sexpr:
     { [ member [ , member . . . ] ] }
     sexpr set - op sexpr              set - op is union diff symdiff inter cross
     opname indexing sexpr             opname is union or inter
     expr .. expr [ by expr ]
     setof indexing member
     if lexpr then sexpr else sexpr
     ( sexpr )
     interval
     infinite - set
     indexing
```

Components of members can be arbitrary constant expressions. Section A.6.3 describes intervals
and infinite - sets.

When used as `binary` operators, union and inter denote the `binary` set operations of union
and intersection. These keywords may also be used as reduction operators.

The .. operator constructs sets. The default by clause is by 1. In general, e 1 ..e 2 by e 3
means the numbers

```ampl
e2 − e1
e 1 , e 1 + e 3 , e 1 + 2e 3 , . . . , e 1 +  ________ e 3
					          e3   
```

rounded to set members. (The notation x denotes the floor of x, that is, the largest integer ≤ x.)
The setof operator is a set construction operator; member is either an expression or a
comma-separated list of expressions enclosed in parentheses. The resulting set consists of all the
members obtained by iterating over the indexing expression; the dimension of the resulting expression
is the number of components in member.

```ampl
abs(x)              absolute value x
acos(x)             inverse cosine, cos − 1 (x)
acosh(x)            inverse hyperbolic cosine, cosh − 1 (x)
alias(v)            alias of model entity v
asin(x)             inverse sine, sin − 1 (x)
asinh(x)            inverse hyperbolic sine, sinh − 1 (x)
atan(x)             inverse tangent, tan − 1 (x)
atan2(y, x)         inverse tangent, tan − 1 (y / x)
atanh(x)            inverse hyperbolic tangent, tanh − 1 (x)
ceil(x)             ceiling of x (next higher integer)
ctime()             current time as a string
ctime(t)            time t as a string
cos(x)              cosine
exp(x)              ex
floor(x)            floor of x (next lower integer)
log(x)              log e (x)
log10(x)            log 10 (x)
max(x, y, ...)      maximum (2 or more arguments)
min(x, y, ...)      minimum (2 or more arguments)
precision(x, n)     x rounded to n significant decimal digits
round(x, n)         x rounded to n digits past decimal point
round(x)            x rounded to an integer
sin(x)              sine
sinh(x)             hyperbolic sine
sqrt(x)             square root
tan(x)              tangent
tanh(x)             hyperbolic tangent
time()              current time in seconds
trunc(x, n)         x truncated to n digits past decimal point
trunc(x)            x truncated to an integer
```

Table A-2: Built-in arithmetic functions.

```ampl
ampl: set y = setof {i in 1..5} (i,iˆ2);
              ampl: display y;
              set y := (1,1) (2,4) (3,9) (4,16) (5,25);
```